In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("Atlantic_Spain.csv")
df.head()

,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url
0,18-05-2024,1,Supernova,Saiko,84,190909,single,1,False,https://i.scdn.co/image/ab67616d0000b273c3f5b9...
1,18-05-2024,2,EL TONTO,Lola Indigo,83,187986,album,11,False,https://i.scdn.co/image/ab67616d0000b273daab36...
2,18-05-2024,3,Acróstico,Shakira,81,170785,single,1,False,https://i.scdn.co/image/ab67616d0000b273bd6bbd...
3,18-05-2024,4,BESO,ROSALÍA & Rauw Alejandro,96,194543,single,3,False,https://i.scdn.co/image/ab67616d0000b2734d6cf0...
4,18-05-2024,5,El Merengue,Marshmello & Manuel Turizo,91,189668,single,1,False,https://i.scdn.co/image/ab67616d0000b273f40467...


In [3]:
entries_per_day = df.groupby('date').size()
entries_per_day.value_counts()

50     554
100      1
Name: count, dtype: int64

In [4]:
entries_per_day[entries_per_day == 100]

date
01-03-2025    100
dtype: int64

In [5]:
df[df['date'] == '2025-03-01']

,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url


In [6]:
df = df.drop_duplicates(subset=['date', 'song'])

In [7]:
entries_per_day = df.groupby('date').size()
entries_per_day.value_counts()

50    542
49     12
51      1
Name: count, dtype: int64

In [8]:
entries_per_day[entries_per_day != 50]

date
01-03-2025    51
01-05-2025    49
02-05-2025    49
03-05-2025    49
04-05-2025    49
05-05-2025    49
06-05-2025    49
07-05-2025    49
08-05-2025    49
09-05-2025    49
28-04-2025    49
29-04-2025    49
30-04-2025    49
dtype: int64

In [9]:
df = df.drop_duplicates(subset=['date', 'song'])

In [10]:
entries_per_day = df.groupby('date').size()
entries_per_day.value_counts()

50    542
49     12
51      1
Name: count, dtype: int64

In [11]:
entries_per_day[entries_per_day == 51]

date
01-03-2025    51
dtype: int64

In [12]:
df[df['date'] == '2025-03-01']

,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url


In [13]:
df[df['date'] == 'YYYY-MM-DD'].groupby('song').size().sort_values(ascending=False)

Series([], dtype: int64)

In [14]:
df = df.sort_values(['date', 'position'])
df = df.groupby('date').head(50)

In [15]:
df['song'].nunique()

545

In [16]:
df['song'] = df['song'].str.strip()
df['artist'] = df['artist'].str.strip()

In [17]:
df['song'] = df['song'].str.lower()
df['artist'] = df['artist'].str.lower()

In [18]:
df['song'] = df['song'].str.replace(r'\s+', ' ', regex=True)
df['artist'] = df['artist'].str.replace(r'\s+', ' ', regex=True)

In [19]:
df['artist'] = df['artist'].str.replace(r'feat\..*', '', regex=True)
df['artist'] = df['artist'].str.replace(r'ft\..*', '', regex=True)
df['artist'] = df['artist'].str.replace(r'featuring.*', '', regex=True)
df['artist'] = df['artist'].str.strip()

In [20]:
df['song'] = df['song'].str.replace(r'\(.*?\)', '', regex=True)
df['song'] = df['song'].str.replace(r'-.*', '', regex=True)
df['song'] = df['song'].str.strip()

In [21]:
df['song'] = df['song'].str.replace(r'\s+', ' ', regex=True)
df['artist'] = df['artist'].str.replace(r'\s+', ' ', regex=True)

In [22]:
df['song'].nunique()

525

In [23]:
df[df['song'].str.contains('remix|live|version', case=False, na=False)][['song']].drop_duplicates().head(20)

,song
12955,una foto remix
15860,trending remix
730,mi nena remix
749,remix exclusivo
21928,wya remix red


In [24]:
df['artist'].nunique()

301

In [25]:
df['artist'].value_counts().head(20)

artist
myke towers                                1229
saiko                                      1223
quevedo                                    1104
karol g                                     922
bad bunny                                   820
mora                                        814
dei v                                       653
feid                                        641
rauw alejandro                              567
manuel turizo                               542
lola indigo                                 464
eladio carrion                              393
sebastian yatra & manuel turizo & beéle     385
dellafuente & morad                         330
marshmello                                  317
saiko & feid & quevedo                      312
gonzy                                       293
rvfv                                        283
emilia                                      277
sky rompiendo & feid & myke towers          271
Name: count, dtype: int64

In [26]:
first_appearance = df.groupby('song')['date'].min().reset_index()
first_appearance.rename(columns={'date': 'entry_date'}, inplace=True)

df = df.merge(first_appearance, on='song', how='left')

In [27]:
lifecycle = df.groupby('song').agg({
    'date': ['min', 'max', 'count'],
    'position': 'min'
}).reset_index()

In [28]:
lifecycle.columns = [
    'song',
    'entry_date',
    'exit_date',
    'days_on_playlist',
    'peak_position'
]

lifecycle.head()

,song,entry_date,exit_date,days_on_playlist,peak_position
0,0 confianza,02-06-2025,23-06-2025,20,23
1,"1 de enero, puntacana",28-04-2025,28-04-2025,1,45
2,1000cosas,01-04-2025,31-10-2025,245,3
3,100xciento,03-05-2025,30-04-2025,11,13
4,14 febreros,24-11-2025,27-11-2025,4,9


In [29]:
peak_dates = df.loc[df.groupby('song')['position'].idxmin()]
peak_dates = peak_dates[['song', 'date']]
peak_dates.rename(columns={'date': 'peak_date'}, inplace=True)

peak_dates.head()

,song,peak_date
5715,0 confianza,07-06-2025
24784,"1 de enero, puntacana",28-04-2025
8193,1000cosas,10-03-2025
9203,100xciento,11-05-2025
21699,14 febreros,24-11-2025


In [30]:
lifecycle = lifecycle.merge(peak_dates, on='song', how='left')

In [31]:
lifecycle['entry_date'] = pd.to_datetime(lifecycle['entry_date'], errors='coerce')
lifecycle['peak_date'] = pd.to_datetime(lifecycle['peak_date'], errors='coerce')

In [32]:
lifecycle[lifecycle['entry_date'].isna()]

,song,entry_date,exit_date,days_on_playlist,peak_position,peak_date
1,"1 de enero, puntacana",NaT,28-04-2025,1,45,NaT
4,14 febreros,NaT,27-11-2025,4,9,NaT
5,2am,NaT,27-11-2025,4,31,NaT
8,4 prez,NaT,29-05-2024,2,35,NaT
16,acho pr,NaT,20-10-2024,7,15,NaT
...,...,...,...,...,...,...
511,whisky,NaT,31-07-2025,15,30,NaT
515,ya es navidad popurrí: ya es navidad,NaT,25-12-2024,1,34,NaT
517,yankee 150,NaT,18-06-2024,1,46,NaT
518,"yes, and?",NaT,20-01-2025,7,30,NaT


In [33]:
lifecycle['time_to_peak'] = (
    lifecycle['peak_date'] - lifecycle['entry_date']
).dt.days

In [34]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['entry_date'] = pd.to_datetime(df['entry_date'], errors='coerce')

In [35]:
df[['date', 'entry_date']].dtypes

date          datetime64[us]
entry_date    datetime64[us]
dtype: object

In [36]:
df['days_since_entry'] = (df['date'] - df['entry_date']).dt.days

In [37]:
df['days_since_entry'] = (df['date'] - df['entry_date']).dt.days

In [38]:
df[['date', 'entry_date', 'days_since_entry']].head()

,date,entry_date,days_since_entry
0,2025-01-01,2025-01-01,0.0
1,2025-01-01,2025-01-01,0.0
2,2025-01-01,2025-01-01,0.0
3,2025-01-01,2025-01-01,0.0
4,2025-01-01,2025-01-01,0.0


In [39]:
def classify_stage(row):
    if row['days_since_entry'] <= 7:
        return 'new'
    elif row['position'] <= 10:
        return 'peak'
    elif row['position'] <= 25:
        return 'growth'
    elif row['position'] <= 40:
        return 'mature'
    else:
        return 'decline'

In [40]:
df.columns

Index(['date', 'position', 'song', 'artist', 'popularity', 'duration_ms',
       'album_type', 'total_tracks', 'is_explicit', 'album_cover_url',
       'entry_date', 'days_since_entry'],
      dtype='str')

In [41]:
df['lifecycle_stage'] = df.apply(classify_stage, axis=1)

In [42]:
df['lifecycle_stage'].value_counts()

lifecycle_stage
mature     7613
growth     7338
decline    5149
peak       4704
new        2934
Name: count, dtype: int64

In [43]:
daily_songs = df.groupby('date')['song'].apply(set)

In [44]:
entries = []
exits = []

dates = sorted(daily_songs.index)

for i in range(1, len(dates)):
    today = daily_songs[dates[i]]
    yesterday = daily_songs[dates[i-1]]

    entries.append(len(today - yesterday))
    exits.append(len(yesterday - today))

In [45]:
churn_df = pd.DataFrame({
    'date': dates[1:],
    'entries': entries,
    'exits': exits
})

churn_df.head()

,date,entries,exits
0,2024-01-07,14,14
1,2024-01-08,11,11
2,2024-01-09,22,22
3,2024-01-10,16,16
4,2024-01-11,19,19


In [46]:
churn_df[['entries', 'exits']].mean()

entries    17.0
exits      17.0
dtype: float64

In [47]:
churn_df.sort_values('entries', ascending=False).head()

,date,entries,exits
193,2025-11-01,45,45
182,2025-10-01,45,45
127,2025-05-01,45,45
138,2025-06-01,45,45
149,2025-07-01,45,45


In [48]:
churn_df['entries'].value_counts()

entries
13    29
15    27
12    24
11    18
16    18
14    17
17    15
18    10
10     9
45     9
19     6
9      6
36     6
8      3
23     3
22     2
37     2
20     2
44     2
40     1
41     1
35     1
26     1
25     1
24     1
Name: count, dtype: int64

In [49]:
song_features = df.drop_duplicates('song')[[
    'song',
    'is_explicit',
    'album_type',
    'duration_ms'
]]

In [50]:
lifecycle = lifecycle.merge(song_features, on='song', how='left')

In [51]:
lifecycle.groupby('is_explicit')['days_on_playlist'].mean()

is_explicit
False    55.820069
True     49.177966
Name: days_on_playlist, dtype: float64

In [52]:
lifecycle.groupby('is_explicit')['time_to_peak'].mean()

is_explicit
False    26.057692
True     54.900000
Name: time_to_peak, dtype: float64

In [53]:
lifecycle.groupby('album_type')['days_on_playlist'].mean()

album_type
album     41.658088
single    64.849802
Name: days_on_playlist, dtype: float64

In [54]:
lifecycle.groupby('album_type')['peak_position'].mean()

album_type
album     22.577206
single    20.600791
Name: peak_position, dtype: float64

In [55]:
lifecycle['duration_min'] = lifecycle['duration_ms'] / 60000

In [56]:
lifecycle[['duration_min', 'days_on_playlist']].corr()

,duration_min,days_on_playlist
duration_min,1.000000,0.031503
days_on_playlist,0.031503,1.000000


In [57]:
lifecycle.sort_values('days_on_playlist', ascending=False).head(10)

,song,entry_date,exit_date,days_on_playlist,peak_position,peak_date,time_to_peak,is_explicit,album_type,duration_ms,duration_min
243,la bachata,2025-01-01,31-12-2024,538,11,2025-01-01,0.0,False,album,162637,2.710617
155,el merengue,2025-01-01,31-12-2024,487,5,2024-01-06,-361.0,False,album,189668,3.161133
91,columbia,2025-01-01,31-12-2024,463,1,2024-01-08,-359.0,False,single,186000,3.100000
499,vagabundo,2025-01-01,31-12-2024,388,3,2024-04-06,-270.0,False,single,215600,3.593333
247,la falda,2025-01-01,31-12-2024,359,1,NaT,NaN,True,album,174229,2.903817
286,luna,2025-01-01,31-12-2024,357,2,2025-11-01,304.0,False,album,196800,3.280000
377,polaris,2025-01-01,31-12-2024,349,1,2024-11-06,-56.0,True,single,288000,4.800000
388,qlona,2025-01-01,31-12-2024,342,8,NaT,NaN,True,album,172797,2.879950
299,manos rotas,2025-01-01,31-12-2024,329,1,2024-01-12,-355.0,False,single,168594,2.809900
269,lala,2025-01-01,31-12-2024,310,1,2024-05-07,-239.0,True,album,197920,3.298667


In [58]:
lifecycle.sort_values('time_to_peak').head(10)

,song,entry_date,exit_date,days_on_playlist,peak_position,peak_date,time_to_peak,is_explicit,album_type,duration_ms,duration_min
155,el merengue,2025-01-01,31-12-2024,487,5,2024-01-06,-361.0,False,album,189668,3.161133
41,baby hello,2025-01-01,31-12-2024,296,2,2024-01-07,-360.0,True,album,222461,3.707683
91,columbia,2025-01-01,31-12-2024,463,1,2024-01-08,-359.0,False,single,186000,3.100000
408,reina,2025-01-01,31-12-2024,229,1,2024-01-09,-358.0,True,album,211879,3.531317
442,si no estás,2025-01-01,31-12-2024,187,1,2024-01-10,-357.0,False,single,184061,3.067683
284,luces de tecno,2025-01-01,31-12-2024,124,7,2024-01-10,-357.0,False,album,164293,2.738217
279,lollipop,2025-01-01,31-12-2024,105,1,2024-01-11,-356.0,True,album,199946,3.332433
299,manos rotas,2025-01-01,31-12-2024,329,1,2024-01-12,-355.0,False,single,168594,2.809900
86,clavaito,2025-01-01,31-08-2024,153,6,2024-02-06,-330.0,False,single,162740,2.712333
322,modelito,2025-01-01,31-12-2024,279,4,2024-02-12,-324.0,True,album,190496,3.174933


In [59]:
lifecycle.sort_values('time_to_peak').head(10)

,song,entry_date,exit_date,days_on_playlist,peak_position,peak_date,time_to_peak,is_explicit,album_type,duration_ms,duration_min
155,el merengue,2025-01-01,31-12-2024,487,5,2024-01-06,-361.0,False,album,189668,3.161133
41,baby hello,2025-01-01,31-12-2024,296,2,2024-01-07,-360.0,True,album,222461,3.707683
91,columbia,2025-01-01,31-12-2024,463,1,2024-01-08,-359.0,False,single,186000,3.100000
408,reina,2025-01-01,31-12-2024,229,1,2024-01-09,-358.0,True,album,211879,3.531317
442,si no estás,2025-01-01,31-12-2024,187,1,2024-01-10,-357.0,False,single,184061,3.067683
284,luces de tecno,2025-01-01,31-12-2024,124,7,2024-01-10,-357.0,False,album,164293,2.738217
279,lollipop,2025-01-01,31-12-2024,105,1,2024-01-11,-356.0,True,album,199946,3.332433
299,manos rotas,2025-01-01,31-12-2024,329,1,2024-01-12,-355.0,False,single,168594,2.809900
86,clavaito,2025-01-01,31-08-2024,153,6,2024-02-06,-330.0,False,single,162740,2.712333
322,modelito,2025-01-01,31-12-2024,279,4,2024-02-12,-324.0,True,album,190496,3.174933


In [60]:
lifecycle.columns

Index(['song', 'entry_date', 'exit_date', 'days_on_playlist', 'peak_position',
       'peak_date', 'time_to_peak', 'is_explicit', 'album_type', 'duration_ms',
       'duration_min'],
      dtype='str')

In [61]:
popularity = df.groupby('song')['popularity'].mean().reset_index()

lifecycle = lifecycle.merge(popularity, on='song', how='left')

In [62]:
lifecycle[['popularity', 'days_on_playlist']].isna().sum()

popularity          0
days_on_playlist    0
dtype: int64

In [63]:
lifecycle = lifecycle.dropna(subset=['popularity', 'days_on_playlist'])

In [64]:
lifecycle[['popularity', 'days_on_playlist']].corr()


,popularity,days_on_playlist
popularity,1.000000,0.358392
days_on_playlist,0.358392,1.000000


In [65]:
lifecycle[['popularity', 'time_to_peak']].corr()

,popularity,time_to_peak
popularity,1.00000,-0.01233
time_to_peak,-0.01233,1.00000


In [66]:
lifecycle.sort_values('popularity', ascending=False).head(10)

,song,entry_date,exit_date,days_on_playlist,peak_position,peak_date,time_to_peak,is_explicit,album_type,duration_ms,duration_min,popularity
492,un x100to,2024-01-06,31-05-2024,27,31,NaT,NaN,False,single,194563,3.242717,99.962963
133,die with a smile,2025-01-10,31-10-2025,48,29,NaT,NaN,False,single,251667,4.194450,99.770833
22,all i want for christmas is you,2025-01-01,27-12-2024,20,1,NaT,NaN,False,album,241106,4.018433,99.000000
244,la bebe,2024-01-06,31-07-2024,104,10,2024-01-06,0.0,True,single,234352,3.905867,98.365385
416,rockin' around the christmas tree,NaT,27-12-2024,4,7,NaT,NaN,False,album,126266,2.104433,98.000000
227,i like the way you kiss me,2025-02-05,27-04-2025,21,19,NaT,NaN,False,single,142514,2.375233,97.857143
169,espresso,2025-01-07,30-06-2025,24,39,NaT,NaN,True,single,175459,2.924317,96.291667
426,santa tell me,NaT,27-12-2024,3,17,NaT,NaN,False,single,204093,3.401550,96.000000
510,where she goes,2024-01-06,31-08-2024,154,1,2024-01-06,0.0,True,single,231704,3.861733,94.512987
324,monaco,2024-01-11,31-12-2024,78,1,NaT,NaN,True,album,267194,4.453233,94.461538


In [67]:
lifecycle.sort_values('days_on_playlist', ascending=False).head(10)

,song,entry_date,exit_date,days_on_playlist,peak_position,peak_date,time_to_peak,is_explicit,album_type,duration_ms,duration_min,popularity
243,la bachata,2025-01-01,31-12-2024,538,11,2025-01-01,0.0,False,album,162637,2.710617,85.635688
155,el merengue,2025-01-01,31-12-2024,487,5,2024-01-06,-361.0,False,album,189668,3.161133,82.303901
91,columbia,2025-01-01,31-12-2024,463,1,2024-01-08,-359.0,False,single,186000,3.100000,89.285097
499,vagabundo,2025-01-01,31-12-2024,388,3,2024-04-06,-270.0,False,single,215600,3.593333,84.804124
247,la falda,2025-01-01,31-12-2024,359,1,NaT,NaN,True,album,174229,2.903817,87.052925
286,luna,2025-01-01,31-12-2024,357,2,2025-11-01,304.0,False,album,196800,3.280000,90.834734
377,polaris,2025-01-01,31-12-2024,349,1,2024-11-06,-56.0,True,single,288000,4.800000,83.481375
388,qlona,2025-01-01,31-12-2024,342,8,NaT,NaN,True,album,172797,2.879950,89.997076
299,manos rotas,2025-01-01,31-12-2024,329,1,2024-01-12,-355.0,False,single,168594,2.809900,77.680851
269,lala,2025-01-01,31-12-2024,310,1,2024-05-07,-239.0,True,album,197920,3.298667,93.451613


In [68]:
df.to_csv("final_data.csv", index=False)

In [69]:
lifecycle.to_csv("lifecycle.csv", index=False)
churn_df.to_csv("churn.csv", index=False)